In [ ]:
# 2026.01.07 
# 세종_문어에 대해서만 작업 완료.
# 문장과 합해서 100만단위씩 나눠서 저장함. 


In [ ]:

import os
import pandas as pd

def split_by_docu_id_target_rows(
    df: pd.DataFrame,
    out_dir: str,
    target_rows: int = 1_000_000,
    docu_col: str = "docu_id",
    sort_cols=("docu_id", "sen_id", "word_id2"),
    prefix: str = "source_shard",
    encoding: str = "utf-8-sig",
):
    """
    docu_id 단위는 유지하면서, shard당 target_rows 근처가 되도록 저장.
    - docu_id 하나가 target_rows보다 큰 경우: 그 docu_id는 어쩔 수 없이 단독 shard로 저장(경계 유지 우선)
    - 저장 파일명: {prefix}_{idx:02d}_rows{n}.csv
    """
    os.makedirs(out_dir, exist_ok=True)

    # 최소한의 방어: docu_id 없으면 실패
    if docu_col not in df.columns:
        raise ValueError(f"'{docu_col}' column not found in df")

    # 정렬(문서/문장/어절 순서 안정화)
    sort_cols = [c for c in sort_cols if c in df.columns]
    if sort_cols:
        # word_id2가 숫자면 숫자 정렬이 더 안전
        if "word_id2" in sort_cols:
            df = df.copy()
            df["word_id2"] = pd.to_numeric(df["word_id2"], errors="coerce")
        df = df.sort_values(sort_cols).reset_index(drop=True)

    # docu_id별 크기
    sizes = df.groupby(docu_col, sort=False).size().reset_index(name="n_rows")

    shard_ids = []
    cur_sum = 0
    shard_idx = 1

    for docu_id, n in zip(sizes[docu_col], sizes["n_rows"]):
        # docu가 너무 큰 경우: 단독 shard로 강제
        if cur_sum > 0 and cur_sum + n > target_rows:
            shard_idx += 1
            cur_sum = 0
        shard_ids.append(shard_idx)
        cur_sum += int(n)
        # 만약 docu 하나가 target_rows보다 크면, 다음 docu부터는 새 shard로
        if cur_sum >= target_rows:
            shard_idx += 1
            cur_sum = 0

    sizes["shard"] = shard_ids

    # docu_id -> shard 매핑
    shard_map = dict(zip(sizes[docu_col], sizes["shard"]))

    df = df.copy()
    df["__shard__"] = df[docu_col].map(shard_map).astype(int)

    # 저장
    out_files = []
    for shard, part in df.groupby("__shard__", sort=True):
        part = part.drop(columns=["__shard__"])
        nrows = len(part)
        out_path = os.path.join(out_dir, f"{prefix}_{shard:02d}_rows{nrows}.csv")
        part.to_csv(out_path, index=False, encoding=encoding)
        out_files.append(out_path)

    # 요약 리포트
    report = (
        df.groupby("__shard__", sort=True)
          .size()
          .reset_index(name="rows")
    )
    return out_files, report


In [ ]:
# 2026.01.07 바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.csv 실행, 
# out_dir: "..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치"

WORD_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver8.csv"   # 네 원본(유지할 18개 컬럼만 있는 파일)
df = pd.read_csv(WORD_CSV_PATH)

OUT_DIR = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치"     # Windows면 raw string 추천
TARGET = 1_000_000  # 문서 경계 우선, TARGET 근처로
if 'file_id' not in df.columns and 'docu_id' in df.columns:
    df = df.rename(columns={'docu_id': 'file_id'})

#optional: 문장 정보 병합
if True:
    SEN_CSV = r"..\csv\original_csv\바른형태분석_세종_문어_형태분석_말뭉치_word_ver7_sen.csv"     
    df_sen = pd.read_csv(SEN_CSV, low_memory=False)

    if 'file_id' not in df_sen.columns and 'docu_id' in df_sen.columns:
        df_sen = df_sen.rename(columns={'docu_id': 'file_id'})
    if 's_form' not in df_sen.columns and 'word_form' in df_sen.columns:
        df_sen = df_sen.rename(columns={'word_form': 's_form'})

    df = df.merge(
        df_sen[["file_id", "sen_id", 's_form', 'sen_len']],
        on=["file_id", "sen_id"],
        how="left",
    )


In [ ]:

out_files, report = split_by_docu_id_target_rows(
    df,
    out_dir=OUT_DIR,
    target_rows=TARGET,
    docu_col="file_id",
    sort_cols=("file_id", "sen_id", "word_id2"),
    prefix="세종문어_source",
)

print("saved:", len(out_files))
display(report)


In [ ]:
# 1) _y를 원래 이름으로 덮어쓰기
for c in df.columns:
    if c.endswith("_y"):
        base = c[:-2]
        df[base] = df[c]

# 2) _x, _y 제거
df = df.drop(columns=[c for c in df.columns if c.endswith("_x") or c.endswith("_y")])
